In [1]:
import sys
import numpy as np
path = '../data/user_movie_rating.npy'

sys.path.append('..')

data = np.load(path)

In [2]:
print("=== SAMPLE DATA ===")
print(data[:5])
# format: [[user_id, movie_id, rating]]

print("=== DIMENSIONS ===")
print(f"Number of users: {len(np.unique(data[:, 0]))}")
print(f"Number of movies: {len(np.unique(data[:, 1]))}")

=== SAMPLE DATA ===
[[  1  30   3]
 [  1 157   3]
 [  1 173   4]
 [  1 175   5]
 [  1 191   2]]
=== DIMENSIONS ===
Number of users: 103703
Number of movies: 17770


In [3]:
# Jackard similarity of two users
from src.similarity import similarity

user1_movies = data[data[:, 0] == 1][:,1]
user2_movies = data[data[:, 0] == 2][:,1]
print(f"User 1 rated {len(user1_movies)} movies, User 2 rated {len(user2_movies)} movies")

sim = similarity(user1_movies, user2_movies)
print(f"Jackard Similarity between items: {sim:.4f}")


User 1 rated 626 movies, User 2 rated 881 movies
Jackard Similarity between items: 0.2312


In [ ]:
# Brute force experiment
# do 5 minutes of randomly searching users with similarity > 0.5

import time
import random
from collections import defaultdict

# Pre-compute all user movie sets
print("Pre-computing 500 user movie set - takes about 1.5 minutes...")
user_movies_sets = {}
unique_users = np.unique(data[:, 0])

# Sample a subset of users for faster processing
sampled_users = np.random.choice(unique_users, size=min(500, len(unique_users)), replace=False)

for user_id in sampled_users:
    user_movies_sets[user_id] = set(data[data[:, 0] == user_id][:, 1])

print(f"Pre-computed sets for {len(user_movies_sets)} users")

# Fast similarity function for sets
def fast_similarity(set1, set2):
    if not set1 or not set2:
        return 0.0
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union > 0 else 0.0

# Run the search
high_similarity_pairs = []
start_time = time.time()
end_time = start_time + 300  # 5 minutes
comparisons = 0

print("\nStarting optimized 5-minute search...")
user_list = list(user_movies_sets.keys())

while time.time() < end_time:
    user1_id, user2_id = random.sample(user_list, 2)

    sim = fast_similarity(user_movies_sets[user1_id], user_movies_sets[user2_id])
    comparisons += 1

    if sim > 0.5: # with 0.5 i usually get 0 similarity but it works with 0.3
        high_similarity_pairs.append((user1_id, user2_id, sim))
        print(f"Found: Users {user1_id} & {user2_id} = {sim:.4f}")

    if comparisons % 1_000_000 == 0:
        print(f"Processed {comparisons:,} pairs...")

print(f"\nCompleted {comparisons:,} comparisons")
print(f"Found {len(high_similarity_pairs)} pairs with similarity > 0.5")

Pre-computing 500 user movie set - takes about 1.5 minutes...
Pre-computed sets for 500 users

Starting optimized 5-minute search...
Found: Users 18410 & 18542 = 0.3068
Found: Users 48778 & 37660 = 0.3329
Found: Users 47915 & 86905 = 0.3001
Found: Users 101917 & 82752 = 0.3303
Found: Users 20485 & 1101 = 0.3022
Found: Users 21002 & 63295 = 0.3057
Found: Users 49987 & 18542 = 0.3016
Found: Users 52522 & 50926 = 0.3529
Found: Users 30725 & 103249 = 0.3184
Found: Users 26697 & 50926 = 0.3015
Found: Users 85895 & 71246 = 0.3150
Found: Users 27695 & 47915 = 0.3067
Found: Users 86905 & 95478 = 0.3181
Found: Users 30125 & 98506 = 0.3240
Found: Users 32589 & 103249 = 0.3081
Found: Users 68788 & 47915 = 0.3002
Found: Users 61515 & 18410 = 0.3157
Found: Users 91024 & 65027 = 0.3111
Found: Users 91343 & 61176 = 0.3105
Found: Users 37236 & 35067 = 0.3354
Found: Users 48778 & 72959 = 0.3044
Found: Users 61176 & 49987 = 0.3263
Found: Users 100998 & 39377 = 0.3079
Found: Users 56761 & 61515 = 0.3029
